# Positional Encoding: Teaching Transformers Word Order

Transformers process all words **simultaneously** — which is fast, but means they have no idea what order the words are in! 

In this notebook, we'll:

1. Understand **why** position information is needed
2. Implement **sinusoidal positional encoding** from scratch
3. **Visualize** the encoding patterns
4. Explore **learned** and **relative** positional encodings
5. See how position encoding gets **added** to word embeddings

**Prerequisites:** [Attention Mechanisms](./01_attention_mechanisms.ipynb) and [Multi-Head Attention](./02_multi_head_attention.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 12

np.random.seed(42)
print("Setup complete!")

## Part 1: The Problem — Why Order Matters

Without positional encoding, these two sentences look identical to a transformer:

In [ ]:
# Demonstrate why position matters
sentence1 = ["The", "dog", "chased", "the", "cat"]
sentence2 = ["The", "cat", "chased", "the", "dog"]

# Simple embeddings (same word = same embedding)
word_embeddings = {
    "The": np.array([1.0, 0.0, 0.0]),
    "the": np.array([1.0, 0.0, 0.0]),
    "dog": np.array([0.0, 1.0, 0.3]),
    "cat": np.array([0.0, 0.8, 0.5]),
    "chased": np.array([0.0, 0.0, 1.0]),
}

emb1 = set(tuple(word_embeddings[w]) for w in sentence1)
emb2 = set(tuple(word_embeddings[w]) for w in sentence2)

print(f'Sentence 1: "{" ".join(sentence1)}"')
print(f'Sentence 2: "{" ".join(sentence2)}"')
print(f"\nSame words?  {'Yes' if emb1 == emb2 else 'No'}")
print(f"Same meaning? No! In sentence 1, the DOG chases. In sentence 2, the CAT chases.")
print(f"\nWithout positional encoding, attention treats these identically.")
print(f"We need to tell the model WHERE each word is!")

## Part 2: Sinusoidal Positional Encoding

The original transformer paper uses **sine and cosine waves** at different frequencies.

### The Clock Analogy

A clock tells time using multiple hands at different speeds:
- **Hour hand** (slow): rough time
- **Minute hand** (medium): specific minute  
- **Second hand** (fast): exact second

Similarly, positional encoding uses waves at different speeds to create a unique pattern for each position.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """
    Create sinusoidal positional encoding matrix.
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    
    Args:
        max_len: Maximum sequence length
        d_model: Embedding dimension
    
    Returns:
        PE matrix of shape (max_len, d_model)
    """
    PE = np.zeros((max_len, d_model))
    
    for pos in range(max_len):
        for i in range(0, d_model, 2):  # Step by 2 (pairs of sin/cos)
            # The denominator controls the frequency
            denominator = 10000 ** (i / d_model)
            
            PE[pos, i]     = np.sin(pos / denominator)  # Even dimensions: sine
            if i + 1 < d_model:
                PE[pos, i + 1] = np.cos(pos / denominator)  # Odd dimensions: cosine
    
    return PE

# Create positional encodings
max_len = 50   # Up to 50 positions
d_model = 16   # 16 dimensions (small for visualization)

PE = sinusoidal_positional_encoding(max_len, d_model)

print(f"Positional encoding matrix shape: {PE.shape}")
print(f"  {max_len} positions × {d_model} dimensions\n")
print("First 5 positions (showing first 8 dimensions):")
print(f"{'Pos':>4}", end="")
for d in range(8):
    label = f"sin_{d//2}" if d % 2 == 0 else f"cos_{d//2}"
    print(f"{label:>8}", end="")
print()
print("─" * 68)
for pos in range(5):
    print(f"{pos:>4}", end="")
    for d in range(8):
        print(f"{PE[pos, d]:>8.3f}", end="")
    print()

## Part 3: Visualize the Encoding as a Heatmap

The famous positional encoding heatmap — each row is a position, each column is a dimension:

In [ ]:
# Larger encoding for better visualization
PE_large = sinusoidal_positional_encoding(100, 64)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(PE_large, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)

ax.set_xlabel('Encoding Dimension', fontsize=13)
ax.set_ylabel('Word Position', fontsize=13)
ax.set_title('Sinusoidal Positional Encoding\n'
             '(each row = unique position fingerprint)', fontsize=14)

plt.colorbar(im, ax=ax, label='Encoding Value')

# Add annotations
ax.annotate('Slow waves\n(big picture)', xy=(2, 90), fontsize=11,
            ha='center', color='black', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
ax.annotate('Fast waves\n(fine detail)', xy=(58, 90), fontsize=11,
            ha='center', color='black', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

print("Left side: slow-changing waves (tell you roughly where you are)")
print("Right side: fast-changing waves (tell you exactly where you are)")
print("Together: every position has a UNIQUE pattern!")

## Part 4: Visualize Individual Waves

Let's look at individual dimensions to see the sine/cosine waves at different frequencies:

In [ ]:
PE_wave = sinusoidal_positional_encoding(100, 64)
positions = np.arange(100)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

dims_to_show = [0, 4, 16, 60]  # Different frequencies
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for ax, dim, color in zip(axes, dims_to_show, colors):
    freq_label = "SLOW" if dim < 8 else "MEDIUM" if dim < 32 else "FAST"
    wave_type = "sin" if dim % 2 == 0 else "cos"
    
    ax.plot(positions, PE_wave[:, dim], color=color, linewidth=2)
    ax.set_ylabel('Value', fontsize=11)
    ax.set_title(f'Dimension {dim} ({wave_type}, {freq_label} frequency)', fontsize=12)
    ax.set_ylim(-1.2, 1.2)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Word Position', fontsize=13)
fig.suptitle('Positional Encoding: Waves at Different Frequencies',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Slow dimensions: change gradually → rough position (beginning/middle/end)")
print("Fast dimensions: change rapidly  → exact position")
print("\nLike a clock: hour hand (slow) + minute hand (medium) + second hand (fast)")
print("= unique time for every moment!")

## Part 5: Every Position Is Unique

Let's verify that each position gets a truly unique encoding by measuring distances between position encodings:

In [ ]:
# Compute distances between all pairs of position encodings
PE_dist = sinusoidal_positional_encoding(50, 64)

# Euclidean distance between each pair of positions
n_pos = 50
distances = np.zeros((n_pos, n_pos))
for i in range(n_pos):
    for j in range(n_pos):
        distances[i, j] = np.linalg.norm(PE_dist[i] - PE_dist[j])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distance matrix
im1 = axes[0].imshow(distances, cmap='viridis', aspect='auto')
axes[0].set_xlabel('Position', fontsize=12)
axes[0].set_ylabel('Position', fontsize=12)
axes[0].set_title('Distance Between Position Encodings\n(darker = more similar)', fontsize=13)
plt.colorbar(im1, ax=axes[0], label='Euclidean Distance')

# Distance from position 0
axes[1].plot(range(n_pos), distances[0], 'b-', linewidth=2, label='From position 0')
axes[1].plot(range(n_pos), distances[10], 'r-', linewidth=2, label='From position 10')
axes[1].plot(range(n_pos), distances[25], 'g-', linewidth=2, label='From position 25')
axes[1].set_xlabel('Position', fontsize=12)
axes[1].set_ylabel('Distance', fontsize=12)
axes[1].set_title('Distance from Reference Positions\n(nearby positions are more similar)', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("1. The diagonal is 0 (each position is identical to itself)")
print("2. Nearby positions have SMALLER distances (more similar)")
print("3. Far-apart positions have LARGER distances (more different)")
print("4. The smooth gradient means the model can learn relative distance!")

## Part 6: Adding Position to Meaning

Now let's see how positional encoding gets **added** to word embeddings in practice:

In [ ]:
# A real-ish example
sentence = ["The", "cat", "sat", "on", "the", "mat"]
d_model = 8

# Word embeddings (random, but in real models these are learned)
np.random.seed(42)
word_emb = np.random.randn(len(sentence), d_model) * 0.5

# Positional encoding
PE_small = sinusoidal_positional_encoding(len(sentence), d_model)

# Add them together!
combined = word_emb + PE_small

# Visualize all three
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, data, title in [
    (axes[0], word_emb, 'Word Embeddings\n(WHAT the word means)'),
    (axes[1], PE_small, 'Positional Encoding\n(WHERE the word is)'),
    (axes[2], combined, 'Combined (Word + Position)\n(WHAT + WHERE)'),
]:
    im = ax.imshow(data, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
    ax.set_yticks(range(len(sentence)))
    ax.set_yticklabels(sentence, fontsize=12)
    ax.set_xlabel('Dimension', fontsize=11)
    ax.set_title(title, fontsize=13)

plt.colorbar(im, ax=axes[2])
fig.suptitle('How Position Gets Added to Meaning', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Formula: final_embedding = word_embedding + positional_encoding')
print()
print('Notice in the middle plot: the two "the" words at positions 0 and 4')
print('have DIFFERENT positional encodings, so even though they are the same word,')
print('they end up with DIFFERENT combined representations (right plot).')

## Part 7: Same Word, Different Positions

Let's prove that positional encoding makes the same word distinguishable at different positions:

In [ ]:
# "the" appears at position 0 and position 4
d_model = 32
PE_check = sinusoidal_positional_encoding(10, d_model)

# Same word embedding for "the"
the_embedding = np.random.randn(d_model) * 0.5

# Add positional encoding at different positions
the_at_pos0 = the_embedding + PE_check[0]
the_at_pos4 = the_embedding + PE_check[4]

# How similar are they?
dot_product = np.dot(the_at_pos0, the_at_pos4)
cos_sim = dot_product / (np.linalg.norm(the_at_pos0) * np.linalg.norm(the_at_pos4))
distance = np.linalg.norm(the_at_pos0 - the_at_pos4)

print('Word: "the" at two different positions')
print(f'  Without positional encoding: identical (distance = 0.0)')
print(f'  With positional encoding:')
print(f'    Euclidean distance: {distance:.4f} (NOT zero — they are distinguishable!)')
print(f'    Cosine similarity:  {cos_sim:.4f} (similar but not identical)')
print(f'\n  The model can now tell that "the" at position 0 (before "cat")')
print(f'  is different from "the" at position 4 (before "mat")!')

# Compare with a completely different word
cat_embedding = np.random.randn(d_model) * 0.5
cat_at_pos1 = cat_embedding + PE_check[1]
distance_diff = np.linalg.norm(the_at_pos0 - cat_at_pos1)
cos_sim_diff = np.dot(the_at_pos0, cat_at_pos1) / (
    np.linalg.norm(the_at_pos0) * np.linalg.norm(cat_at_pos1))

print(f'\n  For comparison, "the"(pos 0) vs "cat"(pos 1):')
print(f'    Distance: {distance_diff:.4f}, Cosine sim: {cos_sim_diff:.4f}')
print(f'    Different words are even more different than same word at different positions.')

## Part 8: Learned Positional Embeddings

An alternative approach used by BERT, GPT-2, GPT-3: **just learn the position encodings from data!**

In [ ]:
class LearnedPositionalEncoding:
    """Position encodings that are learned as parameters during training."""
    
    def __init__(self, max_len, d_model):
        # Initialize randomly (will be updated by gradient descent during training)
        self.embeddings = np.random.randn(max_len, d_model) * 0.02
        self.max_len = max_len
        self.d_model = d_model
    
    def __call__(self, seq_len):
        if seq_len > self.max_len:
            raise ValueError(f"Sequence length {seq_len} exceeds max {self.max_len}!")
        return self.embeddings[:seq_len]


# Compare sinusoidal vs learned
max_len = 50
d_model = 32

sinusoidal_pe = sinusoidal_positional_encoding(max_len, d_model)
learned_pe = LearnedPositionalEncoding(max_len, d_model)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

im1 = axes[0].imshow(sinusoidal_pe, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[0].set_title('Sinusoidal (formula-based)\nUsed by: Original Transformer', fontsize=13)
axes[0].set_xlabel('Dimension', fontsize=11)
axes[0].set_ylabel('Position', fontsize=11)

im2 = axes[1].imshow(learned_pe(max_len), cmap='RdBu_r', aspect='auto', vmin=-0.1, vmax=0.1)
axes[1].set_title('Learned (random initialization)\nUsed by: BERT, GPT-2, GPT-3', fontsize=13)
axes[1].set_xlabel('Dimension', fontsize=11)
axes[1].set_ylabel('Position', fontsize=11)

plt.tight_layout()
plt.show()

print("Sinusoidal: structured pattern from a formula (works for any length)")
print("Learned: random noise at initialization (refined during training)")
print()
print("Trade-offs:")
print("  Sinusoidal: No extra parameters, works for unseen lengths")
print("  Learned:    Extra parameters, but can capture task-specific patterns")
print("  In practice: similar performance. Most modern models use learned.")

## Part 9: The Relative Distance Property

A key advantage of sinusoidal encoding: the model can learn relative distances.

For sinusoidal PE, the dot product between two position encodings depends primarily on their **distance**, not their absolute positions:

In [ ]:
PE_rel = sinusoidal_positional_encoding(100, 128)

# Compute dot products between positions at various distances
# For pairs at the same distance, the dot product should be similar

distances_to_check = range(0, 30)
avg_dot_products = []
std_dot_products = []

for dist in distances_to_check:
    dots = []
    for start in range(0, 70):  # Check multiple starting positions
        end = start + dist
        if end < 100:
            dot = np.dot(PE_rel[start], PE_rel[end])
            dots.append(dot)
    avg_dot_products.append(np.mean(dots))
    std_dot_products.append(np.std(dots))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(distances_to_check, avg_dot_products, 'b-o', markersize=5, linewidth=2)
ax.fill_between(distances_to_check,
                np.array(avg_dot_products) - np.array(std_dot_products),
                np.array(avg_dot_products) + np.array(std_dot_products),
                alpha=0.2, color='blue')
ax.set_xlabel('Distance Between Positions', fontsize=13)
ax.set_ylabel('Average Dot Product', fontsize=13)
ax.set_title('Dot Product vs. Distance (Sinusoidal Encoding)\n'
             'Nearby positions are more similar (higher dot product)', fontsize=14)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The dot product decreases smoothly with distance.")
print("This means the model can learn patterns like:")
print("  'Adjectives are usually 1 position before nouns'")
print("  'Verbs are usually 1-2 positions after subjects'")
print("\nThe small standard deviation (shaded area) shows this relationship")
print("is consistent regardless of absolute position — it depends on DISTANCE.")

## Part 10: Complete Pipeline — From Words to Attention-Ready Inputs

Let's put it all together: tokenization → embedding → positional encoding → ready for attention:

In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


# === Complete Pipeline ===

sentence = ["The", "cat", "sat"]
d_model = 16

print("=" * 60)
print("COMPLETE PIPELINE: From words to attention")
print("=" * 60)

# Step 1: Word embeddings
print("\n--- Step 1: Word Embeddings ---")
np.random.seed(42)
word_embeddings = np.random.randn(len(sentence), d_model) * 0.5
print(f"Shape: {word_embeddings.shape}")
for i, word in enumerate(sentence):
    print(f"  '{word}': [{', '.join(f'{x:.2f}' for x in word_embeddings[i][:4])}  ...]")

# Step 2: Positional encoding
print("\n--- Step 2: Positional Encoding ---")
PE = sinusoidal_positional_encoding(len(sentence), d_model)
print(f"Shape: {PE.shape}")
for i in range(len(sentence)):
    print(f"  pos {i}: [{', '.join(f'{x:.2f}' for x in PE[i][:4])}  ...]")

# Step 3: Add them
print("\n--- Step 3: Combined (Word + Position) ---")
X = word_embeddings + PE
print(f"Shape: {X.shape}")
for i, word in enumerate(sentence):
    print(f"  '{word}' @ pos {i}: [{', '.join(f'{x:.2f}' for x in X[i][:4])}  ...]")

# Step 4: Now it's ready for attention!
print("\n--- Step 4: Run Attention ---")
d_k = 8
W_Q = np.random.randn(d_model, d_k) * 0.3
W_K = np.random.randn(d_model, d_k) * 0.3
W_V = np.random.randn(d_model, d_k) * 0.3

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

scores = Q @ K.T / np.sqrt(d_k)
weights = softmax(scores)
output = weights @ V

print(f"Attention weights (who pays attention to whom):")
for i, word in enumerate(sentence):
    print(f"  '{word}': ", end="")
    for j, w2 in enumerate(sentence):
        print(f"  {w2}={weights[i, j]:.3f}", end="")
    print()

print("\nBecause of positional encoding, the model can now distinguish")
print("between the same word at different positions!")

## Summary

### The Problem
Transformers process all words in parallel → they don't know word order.

### The Solution
Add position information to word embeddings:
```
final_embedding = word_embedding + positional_encoding
```

### Three Approaches

| Approach | How | Used By |
|----------|-----|--------|
| Sinusoidal | Sin/cos waves at different frequencies | Original Transformer |
| Learned | Trainable embedding table | BERT, GPT-2, GPT-3 |
| Relative/Rotary | Encode distance, not absolute position | LLaMA, T5, Mistral |

### Key Takeaways

1. Without position info, "dog chased cat" = "cat chased dog" to a transformer
2. Position is **added** to (not concatenated with) word embeddings
3. Sinusoidal encoding uses waves: slow dims for rough position, fast dims for exact position
4. Each position gets a **unique fingerprint** of encoding values
5. Nearby positions are **more similar** → model can learn relative distance
6. Modern models mostly use **learned** or **rotary** encodings

**Next:** [Transformer Block](./04_transformer_block.ipynb) — putting attention, FFN, layer norm, and residual connections together.